# GenLab — Launcher
Plataforma modular para ejecutar modelos open source de IA en Colab.

**Antes de ejecutar:**
1. Runtime → Cambiar tipo de entorno de ejecución → **T4 GPU**.
2. Consigue tu **HF_TOKEN** (ver celda 5) si no lo tienes.
3. Concede permisos de Drive cuando se solicite.

In [ ]:
# 1. Instalar dependencias
!pip install -q torch diffusers transformers huggingface-hub imageio imageio-ffmpeg accelerate psutil pyyaml

In [ ]:
# 2. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clonar repo y cargar genlab
REPO_URL = 'https://github.com/ke1npro/GenLab.git'

import os, sys
GENLAB_SRC = '/content/genlab'

if os.path.isdir(GENLAB_SRC):
    !git -C $GENLAB_SRC pull
else:
    !git clone $REPO_URL $GENLAB_SRC

sys.path.insert(0, f'{GENLAB_SRC}/src')
os.chdir(GENLAB_SRC)

import genlab
print(f'genlab cargado: {genlab.__file__}')

In [ ]:
# 4. Bootstrap — diagnóstico del entorno
from genlab import GenLab
info = GenLab.bootstrap()

# Checklist visual
hw = info['hardware']
print()
print('--- Checklist ---')
print(f'["OK" if hw["has_gpu"] else "XX"] GPU: {hw["gpu"]}')
print(f'["OK" if info["hf_token_ok"] else "XX"] HF_TOKEN: configurado')
print(f'["OK"]          Drive montado')
print(f'["OK"]          genlab importado')

In [ ]:
# 5. Token de Hugging Face
# ¿Dónde conseguirlo?
# 1. Crea cuenta en https://huggingface.co/join
# 2. Ve a https://huggingface.co/settings/tokens → "New token" → Role: Read
# 3. Copia el token (empieza con 'hf_')
# 4. Acerca del modelo CogVideoX:
#    - https://huggingface.co/THUDM/CogVideoX-2b
#    - Debes aceptar los términos de uso UNA SOLA VEZ (botón "Agree and access repository")
#    - Luego el token funcionará para descargar los pesos

from google.colab import userdata
import os
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('[OK] HF_TOKEN cargado desde Secrets de Colab.')
except Exception:
    token = input('Pega tu HF_TOKEN (o Enter para omitir): ').strip()
    if token:
        os.environ['HF_TOKEN'] = token
        print('[OK] HF_TOKEN configurado manualmente.')
    else:
        print('[XX] HF_TOKEN no configurado. La descarga del modelo fallar\u00e1.')

In [ ]:
# 6. Generar video
# Esto descarga el modelo (~15GB), lo carga en GPU y genera el video.
# En T4 puede tomar 10-20 minutos.

print('[..] Descargando modelo CogVideoX-2b (primera vez: ~15GB)...')
print('[..] Cargando en GPU...')

result = GenLab().run(
    task='text_to_video',
    model='cogvideo',
    prompt='Un astronauta montando un caballo en la luna, estilo cinematogr\u00e1fico',
    config={'model': {'steps': 50, 'fps': 8, 'frames': 49}}
)

print(f'[OK] Video generado: {result["video_path"]}')

In [ ]:
# 7. Mostrar resultado
from IPython.display import Video
Video(result['video_path'], width=480)